In [1]:
import torch
from transformers import CLIPTokenizer
import pandas as pd
import textwrap
from datetime import datetime

from made.data_pipeline.utils import set_plotting_configuration
set_plotting_configuration()

import matplotlib.pyplot as plt

from made.data_pipeline.data.datacomp_handler import decode_webdataset, get_next_batch
from made.data_pipeline.utils import collect_tar_files
from made.paths import MADE_PATH
from made.models.meru.nn import model_init, image_to_numpy, unnormalize_image



In [2]:
model, trs = model_init(pretrained=str(MADE_PATH / "models/ckpt.pt"))
model = model.to("cuda").eval()
tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")
references = torch.load(str(MADE_PATH / "models/reference.pt"))
img_ref = references["img"].to("cuda")
txt_ref = references["txt"].to("cuda")

In [ ]:
dataset = iter(decode_webdataset(
    collect_tar_files("/home/leobaro/Downloads/datasets/web/datacomp/full_datacomp_unimodal_vision_filter_output"), # 20220 samples read in 1m 10s
    get_images=True,
    get_captions=True,
    batch_size=50
))
i_dataset = iter(dataset)
curvature = model.curvature.exp()

In [4]:
uid_score = {}

In [ ]:
from made.data_pipeline.filtering_functions.multimodal_filters import _entailment
import torch

def _image_specificity(txt_ref: torch.Tensor, curv: float, image: torch.Tensor):
    txt_ref = txt_ref.to(image.device)
    ient = _entailment(txt_ref, image, curv)
    return ient.mean(dim=0)

def _text_specificity(img_ref: torch.Tensor, curv: float, text: torch.Tensor):    
    img_ref = img_ref.to(text.device)
    tent = _entailment(text, img_ref, curv)
    return tent.mean(dim=1)


In [ ]:
c = 0
while True:
    batch = get_next_batch(i_dataset)
    if batch is None:
        print("Batch is None")
        break
    if c > 0 and c % 1000 == 0:
        print(f"Processed {c} samples. Breaking..")
        break
    uids, images, captions = batch
    try:
        images = torch.stack([trs(im) for im in images]).to("cuda")

        with torch.no_grad():
            encoded_images = model.encode_image(images)

        with torch.no_grad():
            tokenized_captions = tokenizer(captions, return_tensors="pt", padding="max_length", truncation=True, max_length=77)["input_ids"].to("cuda")
            encoded_captions = model.encode_text(tokenized_captions)

        image_specificity_scores = _image_specificity(txt_ref=encoded_captions, curv=curvature, image=encoded_images)
        text_specificity_scores = _text_specificity(img_ref=encoded_images, curv=curvature, text=encoded_captions)

        for uid, image, img_ss, caption, txt_ss in zip(uids, images, image_specificity_scores, captions, text_specificity_scores):
            uid_score[uid] = (img_ss, txt_ss, uid, image, caption)
        c += len(uids)
        print(f"Processed {c} samples")
    except Exception as e:
        print(e)
        continue


In [ ]:
len(uid_score)

In [ ]:
for uid, tuple_elements in uid_score.items():
    uid_score[uid] = (tuple_elements[0].item(), tuple_elements[1].item(), tuple_elements[2], image_to_numpy(tuple_elements[3]), tuple_elements[4])

In [10]:
image_specificity_scores = {uid: tuple_elements[0] for uid, tuple_elements in uid_score.items()}
text_specificity_scores = {uid: tuple_elements[1] for uid, tuple_elements in uid_score.items()}

In [ ]:
pd.DataFrame(image_specificity_scores.values()).describe()

In [ ]:
pd.DataFrame(text_specificity_scores.values()).describe()

In [ ]:
from made.data_pipeline.utils import set_plotting_configuration
set_plotting_configuration()
import matplotlib.pyplot as plt
import numpy as np
plt.hist(image_specificity_scores.values(), bins=50, label="Image specificity", alpha=0.5)
plt.hist(text_specificity_scores.values(), bins=100, label="Text specificity", alpha=0.5)
plt.title("Specificity scores distribution (DFN)")
plt.xlabel("Specificity score")
plt.ylabel("Frequency")
#plt.axvline(x=np.mean(sim_scores), color='red', linestyle='--', label='mean')
#plt.axvline(x=np.percentile(sim_scores, 25), color='grey', linestyle='--', label='25th and 75th percentiles')
#plt.axvline(x=np.percentile(sim_scores, 75), color='grey', linestyle='--')
plt.legend()
plt.savefig("specificity_scores_distribution.png")
plt.show()


In [ ]:
percentile_ranges = [(x,x+10) for x in range(0, 100, 10)]
percentile_ranges

In [23]:
image_specificity_scores = [tuple_elements[0] for uid, tuple_elements in uid_score.items()]
images = [tuple_elements[3] for uid, tuple_elements in uid_score.items()]

In [24]:
sorted_indices = sorted(range(len(image_specificity_scores)), key=lambda i: image_specificity_scores[i])
sorted_image_specificity_scores = [image_specificity_scores[i] for i in sorted_indices]
sorted_images = [images[i] for i in sorted_indices]

In [ ]:
percentile_ranges_min_max = []
for percentile_range in percentile_ranges:
    p_min, p_max = np.percentile(sorted_image_specificity_scores, percentile_range[0]), np.percentile(sorted_image_specificity_scores, percentile_range[1])
    print(f"{percentile_range[0]}-{percentile_range[1]}", round(p_min, 3), round(p_max, 3))
    percentile_ranges_min_max.append((p_min, p_max))

In [ ]:
percentile_ranges_min_max

In [27]:
def get_samples_and_scores_from_percentile_range(p_min, p_max, scores, samples):
    ok_indexes = [i for i, score in enumerate(scores) if p_min <= score <= p_max]
    return [samples[i] for i in ok_indexes], [scores[i] for i in ok_indexes]

In [ ]:
num_rows = len(percentile_ranges_min_max)
num_cols = 6

plt.rcParams['figure.autolayout'] = False
fig, axes = plt.subplots(num_rows, num_cols, figsize=(15, num_rows*2.2))

for j, percentile_range in enumerate(percentile_ranges_min_max):
    
    samples, scores = get_samples_and_scores_from_percentile_range(
        percentile_range[0], percentile_range[1], sorted_image_specificity_scores, sorted_images)
    

    for i, (image, score) in enumerate(zip(samples[0:num_cols], scores[0:num_cols])):
        axes[j][i].imshow(image)
        #axes[j][i].set_title(f"{round(score, 4)}")
        axes[j][i].axis("off")

#plt.subplots_adjust(hspace=0.3, wspace=0.2)        
plt.tight_layout(pad=0.1)
fig.savefig("images_specificity_percentiles.png", dpi=150)
plt.show()

In [56]:
text_specificity_scores = [tuple_elements[1] for uid, tuple_elements in uid_score.items()]
captions = [tuple_elements[4] for uid, tuple_elements in uid_score.items()]

In [57]:
sorted_indices = sorted(range(len(text_specificity_scores)), key=lambda i: text_specificity_scores[i])
sorted_text_specificity_scores = [text_specificity_scores[i] for i in sorted_indices]
sorted_captions = [captions[i] for i in sorted_indices]

In [ ]:
for caption_score, caption in zip(sorted_text_specificity_scores[0:10], sorted_captions[0:10]):
    print(caption, round(caption_score, 3))

In [ ]:
for caption_score, caption in zip(sorted_text_specificity_scores[-10:], sorted_captions[-10:]):
    print(caption, round(caption_score, 3))

In [14]:
# def find_samples_with_uids(uid_list, n):
#     dataset = iter(decode_webdataset(
#         collect_tar_files("/home/leobaro/Downloads/datasets/web/datacomp/full_datacomp_unimodal_vision_filter_output"), # 20220 samples read in 1m 10s
#         get_images=True,
#         get_captions=True,
#         batch_size=n,
#         valid_uids=uid_list
#     ))
#     uids, images, captions = get_next_batch(iter(dataset))
#     return uids, images, captions


In [ ]:
import numpy as np

percentile_ranges = [(x,x+10) for x in range(0, 100, 10)]
for percentile_range in percentile_ranges:
    p_min, p_max = np.percentile(sorted_image_specificity_scores, percentile_range[0]), np.percentile(sorted_image_specificity_scores, percentile_range[1])
    print(percentile_range[0], p_min, p_max)





In [25]:
data_to_plot = {}

In [ ]:
uid_score

In [17]:
from typing import Dict, List, Tuple
import numpy as np


def get_data_to_plot(uid_score: dict, type_of_sample: str) -> Dict[str, List[Tuple[float, float, str, torch.Tensor, str]]]: 
    """
    Returns a dictionary of data to plot for a given type of sample.
    The dictionary is indexed by the percentile range.
    The value is a list of tuples of the form (image specificity score, text specificity score, uid, image, caption).
    """
    data_to_plot = {}
    for row_i, percentile_range in enumerate(percentile_ranges):

        if type_of_sample == "image":
            specificity_score_index = 0
        elif type_of_sample == "text":
            specificity_score_index = 1
        else:
            raise ValueError(f"Invalid type of sample: {type_of_sample}")

        specificity_scores = [tuple_elements[specificity_score_index] for tuple_elements in uid_score.values()]

        p_min, p_max = np.percentile(specificity_scores, percentile_range[0]), np.percentile(specificity_scores, percentile_range[1])
        print(percentile_range[0], p_min, p_max)

        data = []
        for uid, tuple_elements in uid_score.items():
            if tuple_elements[0] > p_min and tuple_elements[0] < p_max:
                data.append(tuple_elements)
            elif tuple_elements[1] > p_min and tuple_elements[1] < p_max:
                data.append(tuple_elements)
            else:
                continue

        data_sorted_by_score = sorted(data, key=lambda x: x[specificity_score_index])

        data_to_plot[f"{percentile_range}"] = data_sorted_by_score 

    return data_to_plot


In [ ]:
image_data_to_plot = get_data_to_plot(uid_score, "image")

In [ ]:
for p in percentile_ranges:
    first_example = image_data_to_plot[f"({p[0]}, {p[1]})"][0]
    print(first_example[0], first_example[1], first_example[2], first_example[4])

In [ ]:
text_data_to_plot = get_data_to_plot(uid_score, "text")

In [ ]:
for p in percentile_ranges:
    first_example = text_data_to_plot[f"({p[0]}, {p[1]})"][0]
    print(p, first_example[0], first_example[1], first_example[2], first_example[4])

In [ ]:

num_cols = 4
num_rows = len(percentile_ranges)
fig, axes = plt.subplots(num_rows, num_cols, figsize=(5 * num_cols, 5 * num_rows))
#fig.suptitle("DFN similarity scores percentiles", fontsize=16)

row = 0
for percentile_range in image_data_to_plot.keys():

    # percentile_range_data is a list of tuples of the form (image specificity score, text specificity score, uid, image, caption).
    percentile_range_data = image_data_to_plot[percentile_range][:num_cols]

    row_axes = axes[row]
    col = 0
    for image_ss, text_ss, uid, image, caption in percentile_range_data:
        col_ax = row_axes[col] 

        col_ax.imshow(image)
        if len(caption) > 60:
            caption = caption[0:60]
            caption += "..."
        wrapped_caption = "\n".join(textwrap.wrap(caption, width=35))
        col_ax.set_title(f"{wrapped_caption}\n{round(image_ss, 2)}", fontsize=18)
        col_ax.axis("off")
        col += 1
    row += 1
fig.tight_layout()
fig.savefig(f"image_specificity_percentiles_{datetime.now()}.png")

In [ ]:
text_data_to_plot

In [ ]:
for percentile_range in image_data_to_plot.keys():
    uids, images, captions, scores = image_data_to_plot[percentile_range]
    for caption, score in zip(captions, scores):
        print(caption, round(score, 2))
    print()